# Example 1: Symbolic Regressor

In [1]:
import sys
from pathlib import Path

# Add the parent folder (my_project) to Python's search path
sys.path.append(str(Path.cwd().parent))

from sklearn.utils.random import check_random_state
import numpy as np
from sklearn.metrics import r2_score
from joblib import dump, load
from enum import StrEnum
from scipy.stats import spearmanr
from gplearn.genetic_standard import SymbolicRegressor

In [2]:
def generate_benchmark_data(
    name, n_samples, rng, use_gaussian=False, scale_factor=1.0
):
    """Generates benchmark dataset with optional Gaussian distribution sampling."""
    n_features = 5
    shape = (n_samples, n_features)

    # Define domain parameters based on benchmark requirements
    if name == "NGUYEN7":
        # Target interval: [0, 2]
        low, high = 0.0, 2.0
    elif name == "KEIJZER6":
        # Target interval: [1, 100]
        low, high = 1.0, 100.0
    else:
        # Standard interval: [-5, 5]
        low, high = -5.0, 5.0

    if use_gaussian:
        mean = (low + high) / 2.0
        std = ((high - low) / 6.0) * scale_factor
        X = rng.normal(loc=mean, scale=std, size=shape)

        # Truncate out-of-bounds values to preserve domain validity (e.g. x > -1 for log(x+1))
        if name in ["NGUYEN7", "KEIJZER6"]:
            X = np.clip(X, a_min=low, a_max=high)

    else:
        # Uniform Mode (Original behavior)
        X = rng.uniform(low=low, high=high, size=shape)

    return X

In [3]:
def get_benchmark_dataset(name, n_train=100, n_test=100, random_state=0, use_gaussian=False, scale_factor=1.0):
    """
    Generates train and test datasets for standard Genetic Programming benchmarks.

    Parameters:
        name (str): Benchmark problem ('E1', 'E2', 'M1', 'M2', 'H1', 'H2')
        n_train (int): Number of training samples
        n_test (int): Number of testing samples
        random_state (int/RandomState): Seed for reproducibility

    Returns:
        X_train, y_train, X_test, y_test
    """
    rng = check_random_state(random_state)
    name = name.upper().replace("-", "").replace("_", "")

    def _generate_data(n_samples, n_features = 5):

        shape = (n_samples, n_features)

        # Define domain parameters based on benchmark requirements
        if name == "NGUYEN7":
            low, high = 0.0, 2.0
        elif name == "KEIJZER6":
            low, high = 1.0, 100.0
        else:
            low, high = -5.0, 5.0

        if use_gaussian:
            # Gaussian Mode: Center at domain midpoint, set scale to cover domain
            mean = (low + high) / 2.0
            # Set standard deviation so ~99.7% of samples fall within range (3-sigma)
            std = ((high - low) / 6.0) * scale_factor

            X = rng.normal(loc=mean, scale=std, size=shape)

            # Truncate out-of-bounds values to preserve domain validity (e.g. x > -1 for log(x+1))
            if name in ["NGUYEN7", "KEIJZER6"]:
                X = np.clip(X, a_min=low, a_max=high)

        else:
            # Uniform Mode (Original behavior)
            X = rng.uniform(low=low, high=high, size=shape)

        x1, x2, x3, x4, x5 = X[:, 0], X[:, 1], X[:, 2], X[:, 3], X[:, 4]

        # Define Function
        if name == 'KOZA1':
            y = x1**4 + x1**3 + x1**2 + x1
        elif name == 'KOZA2':
            y = x1**5 - 2 * (x1**3) + x1
        elif name == 'KOZA3':
            y = x1**6 - 2 * (x1**4) + x1**2
        elif name == 'NGUYEN1':
            y = x1**3 + x1**2 + x1
        elif name == 'NGUYEN3':
            y = x1**5 + x1**4 + x1**3 + x1**2 + x1
        elif name == 'NGUYEN5':
            y = np.sin(x1**2) * np.cos(x1) - 1
        elif name == 'NGUYEN6':
            y = np.sin(x1) * np.sin(x1 + x1**2)
        elif name == 'NGUYEN7':
            y = np.log(x1 + 1) + np.log(x1**2 + 1)
        elif name == 'NGUYEN10':
            y = np.sin(x1) + np.sin(x2**2)
        elif name == 'PAGIE1':
            y = (1 / (1 + x1**-4)) + (1 / (1 + x2**-4))
        elif name == 'KEIJZER6':
            y = np.array([sum(1.0 / i for i in range(1, int(val) + 1)) for val in x1])
        elif name == 'VLADISLAVLEVA1':
            y = np.exp(-(x1 - 1)**2) * np.cos(x2)
        else:
            raise ValueError(f"Unknown benchmark name: '{name}'.")

        return X, y

    X_train, y_train = _generate_data(n_train)
    X_test, y_test = _generate_data(n_test)

    return X_train, y_train, X_test, y_test

In [4]:
def get_feature_map(function_set, n_features):
    """Creates a dictionary mapping every function name and feature index to a vector slot."""
    feature_map = {}
    idx = 0

    # 1. Map all function names (e.g., 'add', 'sub', 'mul')
    for func in function_set:
        feature_map[func.name] = idx
        idx += 1

    # 2. Map all feature indices (0, 1, 2, ... n_features - 1)
    for feat_idx in range(n_features):
        feature_map[feat_idx] = idx
        idx += 1

    # 3. Map Constant Bins
    constant_bins = [
        'CONST_Q0_To_Q1',  # c < min + 0.25 * range
        'CONST_Q1_To_Q2',  # min + 0.25 * range <= c < min + 0.50 * range
        'CONST_Q2_To_Q3',  # min + 0.50 * range A<= c < min + 0.75 * range
        'CONST_Q3_To_Q4'   # c >= min + 0.75 * range
    ]

    for bin_name in constant_bins:
        feature_map[bin_name] = idx
        idx += 1

    return feature_map

In [5]:
def compute_spearman_scores(y_train, best_programs, X_train):
    """Computes Spearman correlation for best programs per generation without triggering ConstantInputWarning."""
    scores = []

    for program in best_programs:
        y_pred = program.execute(X_train)

        # Check if predictions are constant (zero variance or NaN/Inf issues)
        if np.all(y_pred == y_pred[0]) or np.isnan(y_pred).any():
            scores.append(0.0)
            continue

        # Compute Spearman correlation safely
        stat = spearmanr(y_train, y_pred).statistic

        # Handle potential NaN returns safely
        scores.append(float(np.nan_to_num(stat, nan=0.0)))

    return scores

In [6]:
class MetricKey(StrEnum):
    # Metadata
    seed = 'Seed'
    benchmark = 'Benchmark'
    r2_score_test = 'R2_Score Test'
    r2_score_train = 'R2_Score Train'
    formula = 'Formula'
    program_vector = 'Program Vector'
    parameter = 'Parameter'
    prediction = 'Prediction'
    test_dataset = 'Test Dataset'
    spearman_score = 'Spearman Correlation'

    # Generation Progression
    r2_score_per_gen = 'R2 Score Per Gen'

    # Structural & Fitness Diagnostics
    average_length = 'Average Length'
    average_fitness = 'Average Fitness'
    best_fitness = 'Best Fitness'
    best_length = 'Best Length'

    # Diversity & Distance Metrics
    diversity_distance = 'Diversity Distance'
    diversity_entropy = 'Diversity Entropy'

    # Entropy & Feature Histories
    entropy_probability_history = 'Entropy Probability History'
    entropy_history = 'Entropy History'
    feature_map = 'Feature Map'

In [7]:
# Training Settings

def get_params(random_seed):
    return dict(
    population_size=500,
    generations=200,

    stopping_criteria=0.00,

    tournament_size= 20,
    p_crossover=0.7,
    p_subtree_mutation=0.1,
    p_hoist_mutation=0.05,
    p_point_mutation=0.1,
    max_samples=1.0,
    verbose=1,
    parsimony_coefficient=0.005,
    # parsimony_coefficient=0.1,

    random_state=random_seed,
    metric='rmse',
    function_set = ('add', 'sub', 'mul','div',
                    'sqrt', 'log',
                    'abs', 'neg', 'inv',
                    'max', 'min',
                    'sin', 'cos', 'tan'
                    ),
    feature_names=  ('x1', 'x2', 'x3', 'x4', 'x5'),
    const_range= [2.00, 1.00, 0.50, 0.25, 0.00, -0.25, -0.50, -1.00, -2.00],
    # const_range= (-1.00, 1.00),

    n_elites=1,

    data_record_diversity_distance = True,
        data_record_diversity_entropy = True,

)

In [8]:
# Generate Seed

dataset_size = 100
seed_number = 30
master_rng = np.random.default_rng(seed=42)
random_seeds = master_rng.integers(low=0, high=10000, size= seed_number).tolist()
print(f"Random Seeds = {random_seeds}")

Random Seeds = [892, 7739, 6545, 4388, 4330, 8585, 859, 6973, 2014, 941, 5264, 9756, 7357, 7611, 7174, 7860, 5132, 1281, 8397, 4503, 5003, 3707, 1825, 9267, 7815, 6438, 4024, 8227, 5454, 4434]


In [9]:
def run_training(benchmarks=None, seeds=None, is_gaussian=False, n_train=None):
    results = []

    for seed in seeds:
        print(f"\n==================== SEED {seed} ====================")

        for name in benchmarks:
            # 1. Load dataset using the current seed
            X_train, y_train, X_test, y_test = get_benchmark_dataset(name, random_state=seed, n_train=n_train, use_gaussian=is_gaussian)

            # 2. Instantiate GP with current random_state
            est_gp = SymbolicRegressor(**get_params(seed))
            print(f"\n====================================================================================")
            print(f"Benchmark Running = {name} with Seed {seed}")

            # 3. Train GP
            est_gp.fit(X_train, y_train)
            print(f"Formula: {str(est_gp._program)}")

            # 4. Predict and clean numerical issues
            y_pred_test = est_gp.predict(X_test)
            y_pred_test = np.nan_to_num(y_pred_test, nan=0.0, posinf=1e5, neginf=-1e5)

            y_pred_train = est_gp.predict(X_train)
            y_pred_train = np.nan_to_num(y_pred_train, nan=0.0, posinf=1e5, neginf=-1e5)

            # 5. Calculate Test R2
            r2_test = r2_score(y_test, y_pred_test)
            r2_train = r2_score(y_train, y_pred_train)
            r2_scores = [r2_score(y_train, program.execute(X_train)) for program in
                         est_gp.best_programs_per_gen]
            spearman_scores = compute_spearman_scores(y_train, est_gp.best_programs_per_gen ,X_train)

            # 6. Record findings
            results.append({
                MetricKey.seed: seed,
                MetricKey.benchmark: name,
                MetricKey.r2_score_test: r2_test,
                MetricKey.r2_score_train: r2_train,
                MetricKey.formula: str(est_gp._program),
                MetricKey.program_vector: est_gp._program.similarity_vec,
                MetricKey.spearman_score: spearman_scores,
                MetricKey.prediction: y_pred_train,
                MetricKey.test_dataset: y_test,

                # Data Per Generation
                MetricKey.r2_score_per_gen: r2_scores,
                MetricKey.average_length: est_gp.run_details_['average_length'],
                MetricKey.average_fitness: est_gp.run_details_['average_fitness'],
                MetricKey.best_fitness: est_gp.run_details_['best_fitness'],
                MetricKey.best_length: est_gp.run_details_['best_length'],

                MetricKey.diversity_distance: est_gp.run_details_['diversity_distance'],
                MetricKey.diversity_entropy: est_gp.run_details_['diversity_entropy'],
                MetricKey.entropy_probability_history: est_gp.run_details_['entropy_probability_history'],
                MetricKey.entropy_history: est_gp.run_details_['entropy_history'],
                MetricKey.feature_map: get_feature_map(
                    est_gp._function_set, X_train.shape[1]
                ),
            })

    return results

In [10]:
results = []
benchmarks = [
    # 'koza-1', 'koza-2', 'koza-3',
    'nguyen-1', 'nguyen-3', 'nguyen-5',
    # 'nguyen-6', 'nguyen-7', 'nguyen-10',
    # 'pagie-1', 'keijzer-6', 'vladislavleva-1',
]
results = run_training(benchmarks=benchmarks,seeds=random_seeds,
                       is_gaussian=False, n_train=dataset_size)

# Save Results
filename = 'standard_results_nguyen_1.joblib'
dump(results, f'Graphs/{filename}', compress=3)
print(f"Results successfully saved to '{filename}'")


==================== SEED 892 ====================

Benchmark Running = nguyen-1 with Seed 892
    |   Population Average    |             Best Individual              |
---- ------------------------- ------------------------------------------ ----------
 Gen   Length          Fitness   Length          Fitness      OOB Fitness  Time Left
   0    10.45          54.0323       11          44.6232              N/A      3.48m
   1     3.54          47.9015        6          37.1571              N/A      5.86m
   2     3.87          45.8417        7          10.9787              N/A      3.81m
   3     4.83          42.9232        7          10.9787              N/A      4.88m
   4     7.15          83.1102       11          10.5016              N/A      3.20m
   5     6.63          88.4623       10          10.4996              N/A      2.99m
   6     6.21          68.0717       12      3.04857e-15              N/A      6.79m
   7     8.20          65.4702       12      3.04857e-15        

In [11]:
results = []
benchmarks = [
    # 'koza-1', 'koza-2', 'koza-3',
    # 'nguyen-1', 'nguyen-3', 'nguyen-5',
    'nguyen-6', 'nguyen-7', 'nguyen-10',
    # 'pagie-1', 'keijzer-6', 'vladislavleva-1',
]
results = run_training(benchmarks=benchmarks,seeds=random_seeds,
                       is_gaussian=False, n_train=dataset_size)

# Save Results
filename = 'standard_results_nguyen_2.joblib'
dump(results, f'Graphs/{filename}', compress=3)
print(f"Results successfully saved to '{filename}'")


==================== SEED 892 ====================

Benchmark Running = nguyen-6 with Seed 892
    |   Population Average    |             Best Individual              |
---- ------------------------- ------------------------------------------ ----------
 Gen   Length          Fitness   Length          Fitness      OOB Fitness  Time Left
   0    10.45          12.7095        5         0.534358              N/A     17.85m
   1     7.77          2.16445       18         0.534358              N/A     22.88m
   2     5.23          1.20433       18         0.534358              N/A     18.16m
   3     2.72          1.15566        2         0.534358              N/A     35.84m
   4     1.76           3.6678        2         0.534358              N/A     71.01m
   5     1.28         0.860812        1         0.534358              N/A      6.08m
   6     1.34         0.956963        1         0.534358              N/A      5.73m
   7     1.38          1.35024        1         0.534358        

In [12]:
results = []
benchmarks = [
    # 'koza-1', 'koza-2', 'koza-3',
    # 'nguyen-1', 'nguyen-3', 'nguyen-5',
    # 'nguyen-6', 'nguyen-7', 'nguyen-10',
    'pagie-1', 'keijzer-6', 'vladislavleva-1',
]
results = run_training(benchmarks=benchmarks,seeds=random_seeds,
                       is_gaussian=False, n_train=dataset_size)

# Save Results
filename = 'standard_results_pagie.joblib'
dump(results, f'Graphs/{filename}', compress=3)
print(f"Results successfully saved to '{filename}'")


==================== SEED 892 ====================

Benchmark Running = pagie-1 with Seed 892
    |   Population Average    |             Best Individual              |
---- ------------------------- ------------------------------------------ ----------
 Gen   Length          Fitness   Length          Fitness      OOB Fitness  Time Left
   0    10.45          12.9431        3         0.411663              N/A     13.85m
   1     6.15          1.54719        3         0.411663              N/A     44.94m
   2     3.76          1.09868        2         0.411663              N/A     48.92m
   3     2.72          1.08839        2         0.411663              N/A     35.39m
   4     2.33          1.28708        2         0.411663              N/A      6.84m
   5     2.22         0.902306        2         0.411663              N/A      4.50m
   6     2.23         0.986175        2         0.411663              N/A      3.36m
   7     2.27          1.42313        2         0.411663         